In [ ]:
# Downloading the shakesphere dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
# let us read and inspect it
with open('input.txt', 'r' , encoding = 'utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

In [ ]:
# let us look at the first 1k characters
print(text[:1000])

In [ ]:
# finding out the unique characters in the set
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [ ]:
# let us develop some strategy to tokensize the text - one of out of many tokenization techniques and very simple
# create mapping
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hi there!"))
print(decode(encode("hi there!")))

In [ ]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape , data.dtype)
print(data[:1000]) # lets see how 1st 1k charcters look when rep.as tokens

The bigger the size of each token lesser the size of vocab visavis

In [ ]:
# splitting up the dataset
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

We train the transformer on a tiny little chunk of the dataset not everyting at once


In [ ]:
# we put plus one, to make the amount of learning(8 indv examples packed in) equal to the set block size
block_size = 8
train_data[:block_size+1]

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when in put is {context} the target:{target}")

In [ ]:
torch.manual_seed(1337) # cause it randomly picks up batch_size amount of sequences
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # this is the maximum context length of the predictions

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data # this gives us the data tensor
    # remember train data is also a tensor made a while ago
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

These are 32 input sequences packed into this batch

In [ ]:
print(xb) # this is what we are going to feed into the transformer

In [ ]:
# we will feed it to the simplest language model 
# which is the bigram language model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): # we are predicting what comes next based on the individual identity of a single token
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from this lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets =None):
        logits = self.token_embedding_table(idx) #(Batch, time(context length/block size),length of each embedding)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B*T)
            # in pytorch documentation 
            ''' it expects the dimensions of the logits to be mini batch * channels(which are the dimensions of the tokens)*t
            so we merge them up into 2d tensor, to make those embedding dimensions go into the 2nd dimension of the logits'''
            loss = F.cross_entropy(logits,targets) # simiarly aligning the targets
        return logits , loss
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get the predictions
            logits,loss = self(idx)
            # we will grab out only the last time step, as this is bigram model only-means we need only the character before what we wanna predict
            logits = logits[:,-1,:]
            # apply softmax to get the probs
            probs = F.softmax(logits, dim=-1) # meas what shape we are giving that shape you outpt
            #sample from the distribution
            idx_next = torch.multinomial(probs, num_samples = 1)# this is like a spin wheel with probabailities
            idx = torch.cat((idx, idx_next), dim=1) #(B, T+1)
        return idx
                        
m = BigramLanguageModel(vocab_size)
logits,loss = m(xb, yb)
print(logits.shape)
print(logits)
print(loss) # which should be around 1/65 , expected



In [ ]:
# now lets do some generation with the bigram model we created
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long),max_new_tokens=100)[0].tolist()))

In [ ]:
# create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
# it will take the gradients and update the params acc to the grdients

In [ ]:
batch_size = 32
for steps in range(10000): 

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())


In [ ]:
# we are getting smtg like reasonable-ish
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long),max_new_tokens=500)[0].tolist()))

some layers have diffrent behaviour in training/eval time, keep note of that
plus
estimate_loss -> averages up the loss over multiple batches, we will get avg for both splits
@torch.no_grad() -> we will never need gradients over here, dont keep track

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


### Trick in self attention

In [ ]:
# considering this toy example
torch.manual_seed(1337)
B,T,C = 4,8, 2
x = torch.randn(B,T,C)
x.shape

In [ ]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev,0)

Here we are averaging stuff and putting it into xbow and saving values across time(making them talk to a trivial way basically). so x[b,t,c] is a normal embedding which is formed and xprev is what adds up that particular batch ka embeddings , and we take avg of xprev(in the same incremental fashion) in time dimension as planned above and put it in xbow

In [ ]:
x[0] # first batch of random x

In [ ]:
xbow[0] # first batch of avergaed x

To do the same incremental averaging, we propose having this manipulative matrix a and do it indirectly

In [ ]:
torch.manual_seed(42)
# a = torch.ones(3,3)
a = torch.tril(torch.ones(3,3)) # we are forming the lower traingular matrix out of this
a = a / torch.sum(a, 1, keepdim=True) # sum a in the dimension of 1 and keep the dimensions for braodcasting
b = torch.randint(0,10,(3,2)).float()
c= a @ b # matrix multiplication is being performed
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

So by manipulating the a like how we did above , we can find averages of the matrix in the incremental fashion like we just did above!

In [ ]:
# ver-2 , using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True) # dim needed for future brodcasting
xbow2 = wei @ x # (B,T,T) @ (B,T,C) -> BTC, in wei B is created as its needed and matrix mulplication happens
# checking sanity 
print(torch.allclose(xbow, xbow2, atol=1e-7))

In [ ]:
xbow2[0]  # this looks same as xbow

So we are pulling all these stunts to find weighted aggregates which is gonna help us in future, plus we make sure that a particular token gets info from only the tokens perceiving it

In [ ]:
tril=torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T)) # its begins with zero , but they are going to be dependent , they will look at each other anol
# and it tells us how much from each token in the past do you wanna aggregate on
wei=wei.masked_fill(tril==0,float('-inf')) # cause they arent supposed to see the future
wei=F.softmax(wei,dim=-1)#self attention trick
xbow3=wei@x #(B,T,T) @ (B,T,C) → (B,T,C)
torch.allclose(xbow,xbow3)

**We can do weighted aggregations by using matrix multiplication of lower triangular fashion, the weights say how much of each elements fuses into this aggregation**

In [1]:
# In the script bigram_with_attn.py ,we have added the positional encoding and also added intermediate embedding layer rather than a normal look up
# which basically means creating a level of indirection, which gives the token embeddings
# and we add a linear layer which gives us logits from the token embeddings
# self.lm_head = nn.Linear(n_embd , vocab_size)
# logits = self.lm_head() this give logits , which aint B,T,C but its B,T,vocab_size

In [ ]:
self.position_embedding_table = nn.Embedding(block_size , n_embed)